In [1]:
import numpy as np
import sys
import os
# Get the absolute path of the main directory
main_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
# Add the main directory to sys.path
sys.path.append(main_dir)
from OptFunctions.NonLinearSE import *
from OptFunctions.oneDQHarmOscillatorfunctions import*
from OptFunctions.oneDwellfunctions import *
import time

#Para usar EGNA
from EDAspy.optimization import EGNA
from EDAspy.optimization import UMDAc
from EDAspy.optimization import EMNA

import pickle

In [2]:
def _normalize(dataset:np.array, grid_size: float):
        '''
        Normalizes the solutions i.e. multiply by a constant so that the integral of the solution^2 = 1.
        Be aware that integration depends on problem definition and size!!!!
        '''
        #grid_size = L/variables     dx of integral
        #grid_size = 5/(self.n_variables-1)

        #Assuming a constant size grid we know grid size independently of problem parameters (better code-wise)
        #self.grid_size = grid_X[-1,0] - grid_X[-2, 0]

        for ind in range(len(dataset)):
                integral = 0
                for var in range(dataset.shape[1]):
                    integral = integral + (dataset[ind,:][var]**2)*grid_size ##Tener en cuenta posibles cambios en tamaño grid!!! L caja = 5 

                A = (1/integral)**0.5
                dataset[ind, :] = A * dataset[ind, :]


In [3]:
#Medidas de los experimentos se guardan en un array.shape [Nexp, 9, 5]
#porque se harán Nexp ejecuciones de los problemas
# 9 filas porque tenemos 4 estados de Oscilador, 4 de pozo y 1 de NLSE
# 5 métricas [MSE, E, E%, GP_1000, t_exc] GP_1000 queda desierto en algoritmos diferentes 
# en este orden de manera que por ejemplo los E% de los Nexp del estado Psi0 del pozo es el elemento [:, 4, 2] del array de Resultados

n_variables = [50, 425]#, 800]  #Tamaños del grid (número de dimensiones del problema de optimización)
for vars in n_variables:
    Nexp = 10
    Resultados = np.empty((Nexp, 9, 5))
    ###Limit of iterations for CMAES
    print(f'\n\n########################### Experimentos con {vars} variables ###########################\n\n')
    for experimento in range(Nexp):
        print(f'\n\n########################### Ejecución de Experimentos número: {experimento+1} de {Nexp} ###########################\n\n')
        #################################################################################
        ############## Quantum Harmonic Oscillator 1D ###################################
        #################################################################################
        #First we load the optimized hyperparameters for this problem and this number of variables        
        ruta = f'UMDABO_params/{vars}Osc'
        with open(f'{ruta}.pkl', "rb") as file:
            BO_params = pickle.load(file)

        size_gen = BO_params['size_gen']
        alpha = BO_params['alpha']
        elite_factor = BO_params['elite_factor']

        ### STOP criteria for algorithm ###
        max_f_evals = 1*10**6 ##This is the main criteria to stop algorithm, max_f_evals = max_iter * size_gen.  
        max_iter = int(max_f_evals/size_gen)
        dead_iter = int(max_iter*0.05) #dead_iter = 5% of max_iter

        ##########################################################################################
        ########################### OSCILADOR ARMÓNICO ###########################################
        ##########################################################################################
        print('Quantum Harmonic Oscillator IN PROGRESS\n')
        #Problem grid definition
        X_grid = np.linspace(start=-4, stop=4, num=vars).reshape(-1,1) #para que sean n_variables filas y 1 columna
        #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
        #Los datos que conocemos son las amplitudes en x=-4 (-inf) x=4 (inf) que son 0
        X_train = X_grid[[0,-1]]
        Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array

        #Analytical solutions definitions
        #Ground state
        APsi0 = np.exp(-X_grid**2/2)
        integral =0
        for i in range(len(APsi0)):
            integral = integral + (APsi0[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi0)-1))

        B = (1/integral)**0.5
        APsi0 = B * APsi0

        #1st excitated state
        APsi1 = np.exp(-X_grid**2/2)*2*X_grid
        integral =0
        for i in range(len(APsi1)):
            integral = integral + (APsi1[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi1)-1))

        B = (1/integral)**0.5
        APsi1 = -1 * B * APsi1 #signo menos para correspondencia con solución algoritmo (simetrización)

        #2nd excitated state
        APsi2 = np.exp(-X_grid**2/2)*(4*X_grid**2-2)
        integral =0
        for i in range(len(APsi2)):
            integral = integral + (APsi2[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi2)-1))

        B = (1/integral)**0.5
        APsi2 = B * APsi2

        #3rd excitated state
        APsi3 = np.exp(-X_grid**2/2)*(8*X_grid**3-12*X_grid)
        integral =0
        for i in range(len(APsi3)):
            integral = integral + (APsi3[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi3)-1))

        B = (1/integral)**0.5
        APsi3 = -1 * B * APsi3 #signo menos para correspondencia con solución algoritmo (simetrización)

        APsis = {}
        APsis['APsi0'] = APsi0[:, 0]
        APsis['APsi1'] = APsi1[:, 0]
        APsis['APsi2'] = APsi2[:, 0]
        APsis['APsi3'] = APsi3[:, 0]
        X_grid = X_grid[:, 0]
        grid_size = X_grid[-1] - X_grid[-2]

        ##########################################################################################################
        ##########################################################################################################
        ### Ahora el código con EGNA que calcula los 4 primeros estados excitados del problema oscilador armónico
        params = {'x':X_grid}
        #Ground state
        print(f'F should be approx -0.082')
        qegna = UMDAc(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=alpha, elite_factor=elite_factor)#, lower_factor=0.01)
        #qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype])#, init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultQHOground = qegna.minimize(oneDHOExpE, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 0, 4] = end-start # 5 métricas [MSE, E, E%, GP_1000, t_exc] GP_1000 queda desierto en algoritmos diferentes 
        _normalize(dataset=egnaresultQHOground.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        #Se guardan resultados de algoritmo para siguiente estado excitado
        Psi_ground=egnaresultQHOground.sel_inds_hist[-1, 0, :-1]
        cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
        E0 = np.sqrt(-np.log(-egnaresultQHOground.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así 
        params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground}
        Resultados[experimento, 0, 1] = E0
        Resultados[experimento, 0, 2] = abs(E0-0.5)/0.5*100

        print(f'\n  |Psi0> done\n')

        #1er estado excitado
        print(f'F should be approx -4.54e-5')
        qegna = UMDAc(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=alpha, elite_factor=elite_factor)#, lower_factor=0.01)
        #qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype])#, init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultQHO1 = qegna.minimize(oneDHOExpE1, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 1, 4] = end-start
        _normalize(dataset=egnaresultQHO1.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        Psi1=egnaresultQHO1.sel_inds_hist[-1, 0, :-1]
        cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
        E1 = np.sqrt(-np.log(-egnaresultQHO1.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
        Resultados[experimento, 1, 1] = E0 + E1
        Resultados[experimento, 1, 2] = abs(E0 + E1 -1.5)/1.5*100
        
        params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1}
        print(f'\n  |Psi1> done\n')

        #2o estado excitado
        print(f'F should be approx -4.54e-5')
        qegna = UMDAc(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=alpha, elite_factor=elite_factor)#, lower_factor=0.01)
        #qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype])#, init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultQHO2 = qegna.minimize(oneDHOExpE2, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 2, 4] = end-start
        _normalize(dataset=egnaresultQHO2.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        Psi2=egnaresultQHO2.sel_inds_hist[-1, 0, :-1]
        cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
        E2 = np.sqrt(-np.log(-egnaresultQHO2.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
        Resultados[experimento, 2, 1] = E0 + E1 + E2
        Resultados[experimento, 2, 2] = abs(E0 + E1 + E2 -2.5)/2.5*100
        
        params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2 }
        print(f'\n  |Psi2> done\n')

        #3er estado excitado
        print(f'F should be approx -4.54e-5')
        qegna = UMDAc(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=alpha, elite_factor=elite_factor)#, lower_factor=0.01)
        #qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype])#, init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultQHO3 = qegna.minimize(oneDHOExpE3, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 3, 4] = end-start
        _normalize(dataset=egnaresultQHO3.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        Psi3=egnaresultQHO3.sel_inds_hist[-1, 0, :-1]
        cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
        E3 = np.sqrt(-np.log(-egnaresultQHO3.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
        Resultados[experimento, 3, 1] = E0 + E1 + E2 + E3
        Resultados[experimento, 3, 2] = abs(E0 + E1 + E2 + E3 -3.5)/3.5*100
        
        params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2, 'E3': E3, 'Psi3': Psi3 }
        print(f'\n  |Psi3> done\n')

        #Cálculos de métricas error MSE de cada estado con estado teórico
        MSEs = []
        for i in range(4):
            mse = min(np.mean((params[f'Psi{i}'] - APsis[f'APsi{i}'])**2), np.mean((-1*params[f'Psi{i}'] - APsis[f'APsi{i}'])**2))
            Resultados[experimento, i, 0] = mse # 5 métricas [MSE, E, E%, GP_1000, t_exc] GP_1000 queda desierto en algoritmos diferentes 
            MSEs.append(mse)
        print(f'MSE: {MSEs}\n')
        print('\nInfinite Oscillator DONE\n')

        ######################################################################################
        ######################################################################################
        ############## POZO DE POTENCIAL INFINITO 1D WELL ####################################
        ######################################################################################
        #First we load the optimized hyperparameters for this problem and this number of variables        
        ruta = f'UMDABO_params/{vars}Well'
        with open(f'{ruta}.pkl', "rb") as file:
            BO_params = pickle.load(file)

        size_gen = BO_params['size_gen']
        alpha = BO_params['alpha']
        elite_factor = BO_params['elite_factor']
        ### STOP criteria for algorithm ### max_f_evals is fixed at the beginning  1 x 10^6  
        max_iter = int(max_f_evals/size_gen)
        dead_iter = int(max_iter*0.05) #dead_iter = 5% of max_iter
        

        print('\nInfinite Well IN PROGRESS\n')
        L=5
        #Energías teóricas
        Eg = (2*np.pi)**2/(8*L**2*1)
        En=[]
        for i in range(4):
            e=(i+1)**2*Eg
            En.append(e)

        ## Problem grid definition
        X_grid = np.linspace(start=0, stop=L, num=vars).reshape(-1,1) #para que sean n_variables filas y 1 columna
        #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
        #Los datos que conocemos son las amplitudes en x=0 x=5 que son 0
        X_train = X_grid[[0,-1]]
        Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array
        init_data = None #initialize_population(size_gen=3000, grid=X_grid, init_kernels=10, init_constraints=(X_train, Y_train))
        grid_size = X_grid[-1,0] - X_grid[-2, 0]
        params = {'L':L}

        #Soluciones teóricas
        ### Soluciones analíticas
        # def Psi_n(n):
        #     return((2/L)**0.5*np.sin(n*np.pi*np.linspace(0, L, n_variables)/L))

        APsisWell = {}
        for n in range(4):
            APsisWell[f'APsi{n}'] = (2/L)**0.5*np.sin((n+1)*np.pi*np.linspace(0, L, vars)/L)
        ###############################    
        #Ground state n=1
        ###############################
        print(f'F should be approx -0.9618')
        qegna = UMDAc(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=alpha, elite_factor=elite_factor)#, lower_factor=0.01)
        #qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype], init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultgroundWell = qegna.minimize(oneD_wellExpE, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 4, 4] = end-start # 5 métricas [MSE, E, E%, GP_1000, t_exc] GP_1000 queda desierto en algoritmos diferentes 
        _normalize(dataset=egnaresultgroundWell.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        Psi_fund=egnaresultgroundWell.sel_inds_hist[-1, 0, :-1]
        E_fund = np.sqrt(-np.log(-egnaresultgroundWell.sel_inds_hist[-1,0,-1])) # def exponencial de función de coste la energía se determina a partir del fitness así.
        #En este caso no hay constante multiplicando exponente exponencial
        Resultados[experimento, 4, 1] = E_fund
        Resultados[experimento, 4, 2] = abs(E_fund-En[0])/En[0]*100 

        params = {'L':L, 'E': E_fund, 'Psi': Psi_fund}
        print(f'\n  |Psi0> done\n')

        ###############################   
        #1excitated state n=2
        ###############################   
        print(f'F should be approx -0.7042')
        qegna = UMDAc(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=alpha, elite_factor=elite_factor)#, lower_factor=0.01)
        #qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype], init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultWellexc1 = qegna.minimize(oneD_wellExpEexcitated, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 5, 4] = end-start
        _normalize(dataset=egnaresultWellexc1.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        Psi_1=egnaresultWellexc1.sel_inds_hist[-1, 0, :-1]
        E1 = np.sqrt(-np.log(-egnaresultWellexc1.sel_inds_hist[-1,0,-1]))
        Resultados[experimento, 5, 1] = E_fund + E1
        Resultados[experimento, 5, 2] = abs(E_fund + E1 -En[1])/En[1]*100

        params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1}
        print(f'\n  |Psi1> done\n')

        ###############################   
        #2nd excitated state n=3
        ###############################   
        print(f'F should be approx -0.3775')
        qegna = UMDAc(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=alpha, elite_factor=elite_factor)#, lower_factor=0.01)
        #qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype], init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultWellexc2 = qegna.minimize(oneD_wellExpEexcitated3, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 6, 4] = end - start
        _normalize(dataset=egnaresultWellexc2.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        Psi_2=egnaresultWellexc2.sel_inds_hist[-1, 0, :-1]
        E2 = np.sqrt(-np.log(-egnaresultWellexc2.sel_inds_hist[-1,0,-1]))
        Resultados[experimento, 6, 1] = E_fund + E1 + E2
        Resultados[experimento, 6, 2] = abs(E_fund + E1 +E2 -En[2])/En[2]*100

        params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2}
        print(f'\n  |Psi2> done\n')

        ###############################   
        #3rd excitated state n=4
        ###############################   
        print(f'F should be approx -0.1482')
        qegna = UMDAc(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=alpha, elite_factor=elite_factor)#, lower_factor=0.01)
        #qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype], init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultWellexc3 = qegna.minimize(oneD_wellExpEexcitated4, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 7, 4] = end-start
        _normalize(dataset=egnaresultWellexc3.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        Psi_3 = egnaresultWellexc3.sel_inds_hist[-1, 0, :-1]
        E3 = np.sqrt(-np.log(-egnaresultWellexc3.sel_inds_hist[-1,0,-1]))
        Resultados[experimento, 7, 1] = E_fund + E1 + E2 + E3
        Resultados[experimento, 7, 2] = abs(E_fund + E1 +E2+E3 -En[3])/En[3]*100

        params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2, 'E3':E3, 'Psi3': Psi_3}
        print(f'\n  |Psi3> done\n')

        #Cálculos de métricas error MSE de cada estado con estado teórico
        MSEs = []
        for i in range(4):
            mse = min(np.mean((params[f'Psi{i}'] - APsisWell[f'APsi{i}'])**2), np.mean((-1*params[f'Psi{i}'] - APsisWell[f'APsi{i}'])**2))
            Resultados[experimento, i+4, 0] = mse # 5 métricas [MSE, E, E%, GP_1000, t_exc] GP_1000 queda desierto en algoritmos diferentes 
            MSEs.append(mse)
        print(f'MSE: {MSEs}\n')  
        print('\nInfinite Well DONE\n')

        ######################################################################################
        ######################################################################################
        #################### N L S E Non linear Schroedinger Equation ########################
        ######################################################################################
        #First we load the optimized hyperparameters for this problem and this number of variables        
        ruta = f'UMDABO_params/{vars}NLSE'
        with open(f'{ruta}.pkl', "rb") as file:
            BO_params = pickle.load(file)

        size_gen = BO_params['size_gen']
        alpha = BO_params['alpha']
        elite_factor = BO_params['elite_factor']
        ### STOP criteria for algorithm ### max_f_evals is fixed at the beginning  1 x 10^6  
        max_iter = int(max_f_evals/size_gen)
        dead_iter = int(max_iter*0.05) #dead_iter = 5% of max_iter

        if vars%2 ==0:
            varsNLSE=vars+1
        else:
            varsNLSE=vars
        
        print('\nNLSE IN PROGRESS\n')
        L=2
        X_grid = np.linspace(start=-1, stop=1, num=varsNLSE).reshape(-1,1) #para que sean n_variables filas y 1 columna
        #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
        #Los datos que conocemos son las amplitudes en x=0 que son 1
        #X_train = X_grid[n_variables//2]
        X_train = np.array((0)).reshape(-1,1)
        Y_train = np.linspace(start=1, stop=1, num=1).reshape(-1,1) #Para que sea 2D array
        
        init_data = None #initialize_population(size_gen=3000, grid=X_grid, init_kernels=10, init_constraints=(X_train, Y_train))
        params = {'L':L}
        qegna = UMDAc(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables=varsNLSE, lower_bound=0, upper_bound=2, alpha=alpha, elite_factor=elite_factor)#, lower_factor=0.01)
        #qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= varsNLSE, lower_bound=0, upper_bound = 2, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype], init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= varsNLSE, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= varsNLSE, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egparesultNLSE = qegna.minimize(NLSE, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 8, 4] = end-start # 5 métricas [MSE, E, E%, GP_1000, t_exc]
        MSE_NLSE = np.sum((1/np.cosh(X_grid[:,0]) - egparesultNLSE.sel_inds_hist[-1, 0, :-1])**2) / len(X_grid[:,0])
        MSE_NLSE = np.float64(MSE_NLSE)
        Resultados[experimento, 8, 0] = MSE_NLSE
        #setting non used columns to None
        Resultados[experimento, 8, 1:3] = None
        Resultados[experimento, :, 3] = None
        print(f'MSE: {MSE_NLSE}\n')    
        print('\nNLSE DONE\n')
        print(f'EXPERIMENT {experimento+1} of {Nexp} FINISHED\n')

    rutafinal = f'UMDA_results/{vars}'    
    with open(f'{rutafinal}.pkl', "wb") as file:
        pickle.dump(Resultados, file)



########################### Experimentos con 50 variables ###########################




########################### Ejecución de Experimentos número: 1 de 10 ###########################


Quantum Harmonic Oscillator IN PROGRESS

F should be approx -0.082
IT:  0 	Best cost:  506.38449232005814
	NFEVALS = 155000 F = -5.973317420290782e-140
	X = [[-2.35658762  0.3155131  -0.28401044 -0.03379254 -0.34558793 -0.37523224
   0.11311522  0.4000437   0.44299881  0.2086061   0.42104001  0.09939156
   0.48352012 -0.2369625   0.10933439 -0.4932827   0.26178809  0.08285565
   0.06104221  0.39637903  0.01976135 -0.51576391  0.57924551  0.75674401
   0.08597978  0.32845252 -0.06916558 -0.47808162 -0.6385577  -0.00963547
   0.39876604  0.58157743 -0.04612236 -0.63045342  0.06508486 -0.13599486
  -0.33712497  0.2358164   0.25563741 -0.03203708  0.7069827   0.30869027
   0.01786843 -0.52895664 -0.31060468  0.12619959 -0.39873813 -0.77536598
   0.2559706  -2.72767116]]

  |Psi0> done

F should be app

/tmp/ipykernel_414251/4095254146.py:106: RuntimeWarning: divide by zero encountered in log
  E0 = np.sqrt(-np.log(-egnaresultQHOground.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así


IT:  0 	Best cost:  inf
	NFEVALS = 51000 F = -0.0
	X = [[ 4.86688187e-01  1.23533508e+00 -1.27628120e+00 -1.24049132e+00
   7.59067342e-01  4.96007125e-01  3.96065053e-02  1.16992297e+00
   1.09954581e+00  5.93223128e-02  6.04600784e-01  1.29810816e-01
   5.86531502e-01  8.71266039e-01 -5.21115786e-01 -4.41441441e-01
  -2.89671238e-01  1.05336383e+00 -1.68824043e+00  3.21208769e-01
   4.52322327e-01  1.65424669e-02  1.11225864e+00 -8.11129585e-01
   2.80704198e-01 -4.45776989e-02  9.27763891e-01  1.18080218e+00
  -1.31647099e-01  4.98343041e-01  3.41516132e-01  2.91531577e-01
   4.19977335e-01  2.29557688e-01  5.53388967e-01  1.56378489e-01
  -1.21462965e+00 -4.28908828e-01 -5.73608324e-01 -5.53560439e-01
  -3.39193591e-01 -1.79514252e-01 -5.60838528e-01  1.12576054e-01
  -9.33390865e-01 -1.23298430e+00 -1.13377838e+00 -1.21918599e-01
  -2.65073254e-01  1.26924480e-01 -1.31034182e+00 -3.53848169e-01
   1.11166309e+00 -1.39524621e+00 -1.73572718e-01  7.56878938e-01
  -1.30486024e-01  1.

/tmp/ipykernel_414251/4095254146.py:126: RuntimeWarning: divide by zero encountered in log
  E1 = np.sqrt(-np.log(-egnaresultQHO1.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así


IT:  0 	Best cost:  inf
	NFEVALS = 51000 F = -0.0
	X = [[-1.09035220e+00 -2.24348231e-01  8.01130756e-02 -5.18847317e-01
  -3.36639938e-01 -8.26193223e-01 -3.42428169e-01  2.60225790e-01
   2.58053192e-01  7.69280879e-01 -7.73201741e-01 -8.46196197e-02
  -9.99291246e-01  4.35835506e-01 -1.65789317e-01  8.26262629e-01
   1.92406529e-01 -1.23625387e+00 -3.82958774e-04 -3.90278906e-01
   9.24972227e-02  1.95932682e-01 -8.81633476e-01 -2.19473338e-01
  -1.41551008e+00 -9.09039007e-01 -2.15455089e-01  9.22673090e-01
  -1.31945677e-01 -1.08802724e-01  7.93504844e-01  1.08749258e+00
   1.34891357e+00 -8.34289396e-01 -3.60896659e-02  5.18084284e-01
   9.12190336e-01 -2.74795467e-01 -1.47452066e+00 -1.78134797e-02
  -3.08823249e-01  4.26763365e-01  1.03611669e-01  6.10038683e-01
  -6.44787885e-01 -6.95643636e-01 -1.13783264e+00  3.61047796e-01
   1.20464329e-01 -4.50874901e-01 -9.33605618e-02 -8.13163715e-01
  -1.43986245e-01 -3.62178356e+00 -2.10869866e+00  2.79298103e-01
   1.94789995e-01  9.

/tmp/ipykernel_414251/4095254146.py:146: RuntimeWarning: divide by zero encountered in log
  E2 = np.sqrt(-np.log(-egnaresultQHO2.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así


... (7 outputs left out)

/tmp/ipykernel_414251/4095254146.py:283: RuntimeWarning: divide by zero encountered in log
  E2 = np.sqrt(-np.log(-egnaresultWellexc2.sel_inds_hist[-1,0,-1]))


IT:  0 	Best cost:  inf
	NFEVALS = 70000 F = -0.0
	X = [[ 0.24731584 -0.68860592  0.19424239 -0.01762495  0.19145849 -0.94177268
  -0.67638386 -0.27099908  0.14110556  0.70352447 -0.52378727  0.69304799
  -0.62567912 -0.71947646  0.30630711  0.18617051  0.24946149 -0.31438212
  -1.02787164 -0.17574433  1.15195675 -0.60729994  0.52386961 -0.98662602
  -0.26661965  0.31415821  1.19936474  0.631128    0.10446291 -0.34615774
  -0.34657783  0.03165608  0.10657458  1.34142968  0.24733118  0.10538142
  -0.53661341  0.82470028 -0.22979459 -0.87245516 -0.22655636 -0.17624699
  -0.49558624  0.20034629 -0.3937626  -0.75582351 -0.5856452  -0.5178796
  -0.30099718 -0.47894051 -0.53626026  0.95833025  1.63444062  0.63401779
   0.50080447 -0.90995104 -0.34497255 -0.18787005 -0.96077576  0.61623076
  -0.20781856 -0.47102754  0.63400801 -0.543447   -0.33577926  0.82373729
  -0.09510001  0.95580674 -0.64196699 -0.24241288 -0.22240638  0.25212707
   0.67315965  0.17788741 -0.61445658  0.03859722 -0.84005

/tmp/ipykernel_414251/4095254146.py:304: RuntimeWarning: divide by zero encountered in log
  E3 = np.sqrt(-np.log(-egnaresultWellexc3.sel_inds_hist[-1,0,-1]))


IT:  0 	Best cost:  968850443.3829810655
	NFEVALS = 108000 F = -0.0
	X = [[ 0.97654685  1.62476041  0.86744532  0.05132981  0.85725799  0.40563049
   1.08356686  0.83565821  1.42857094  0.56075356  0.55884023  1.16791561
   0.52074022  0.89440701  1.34231622 -0.02916745  1.22598669  1.41093815
  -0.15552764  0.89067643  1.27824734  0.65765321  1.77836649  1.7657496
   0.93210161  0.85285375  0.59028921  1.89620361  1.0946076   1.34579647
   1.60643153  1.19135377  1.33190475  0.7096789   0.58648139  0.66945342
   0.81155284  0.72900346  0.93013775  0.35947452  0.71080277 -0.02720737
   0.94156392  0.82796109  0.57573196  1.05821371  1.1524922   1.67719508
   1.02804069  0.10423666  0.71617183  0.6045722   1.3706414   0.62653973
   1.32680399  0.8958353   0.97434436  0.99711178  1.22776507  0.04688863
   1.46920634  0.49869552  1.08923302  1.03871574  1.24712116  0.04785961
   0.89278442  0.31696312  2.04871574  0.67821019  1.27909412  1.61314124
   1.4874146   1.1179398   1.16134077  1